# Setup

In [ ]:
#You might need to install these packages before running. Comment out after
!pip install torch
!pip install transformers datasets
!pip install scikit-learn
!pip install wandb

In [1]:
# Run this cell to mount your Google Drive.
# ONLY ON GOOGLE COLAB!
from google.colab import drive
drive.mount('/content/drive')
%cd "/content/drive/My Drive/nlpproject/"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/My Drive/nlpproject


In [11]:
!wandb login

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter, or press ctrl+c to quit: 
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


In [2]:
import wandb
import csv
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForMaskedLM,AutoModelForSequenceClassification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import classification_report, accuracy_score
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [18]:
def multi_label_formatting(data):
    """This function does not really play a huge role for us right now, but what it basically does is that if a paragraph
    has two stances, i.e. 'conservative' and 'right', it will keep both for the classification problem.   """
    labels = []
    for i in range(len(data['stance'])):
        multi_tags = []
        if type(data['stance'][i]) is not float:
            multi_tags.append(data['stance'][i].lower())

        labels.append(multi_tags)

    return labels

#Preprocessing the CSV file that contains the BASIL database.
df = pd.read_csv('processed_data.csv')

paragraphs = df["body"] # Gets paragraphs from CSV

#This line was intended to convert stances to integers. Is not needed anymore since now we use MultiLabel Binarizer for
# multilabel classification
# df['stance'] = df['stance'].replace([ 'left', 'center', 'liberal', 'conservative', 'right'],[0,1,2,3,4])


stances = df['stance'] # Gets all the stances/labels
# Stances: {'left', 'center', 'liberal', 'conservative', 'right'}
df['body'] =df['body'].astype(str)

In [22]:
def init_data_model(batch_size, test_size):

    # Use if you would want to print a sample paragraph and label
    # print(df['body'][100])
    # print(df['stance'][100])

    #This package will convert tags to an array of size 5 (five because we have 5 stances:
    # 'left', 'center', 'liberal', 'conservative', 'right') where, for example, if a paragraph is classified as 'center'
    # it converts its label into one hot encoding [0,0,1,0,0]
    mlb = MultiLabelBinarizer()
    labels = multi_label_formatting(df) # In case of multitags. Look at function description for more info
    print(f"Labels : {labels}")
    #One Hot Enconding of Multi labels
    labels = mlb.fit_transform(labels)

    #Splitting data into test set and training set.
    x_train_og, x_test_og, y_train, y_test = train_test_split(df['body'].astype(str), labels,test_size=test_size, random_state = 0)

    #These following two models are way bigger and perform worse (tested.)
    # model_name = "roberta-large"
    # model_name = "roberta-base"

    model_name = "launch/POLITICS" # POLITICS model from HuggingFace!

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    # You can check that maximum amount of tokes is 512 which means that we will not be able
    # to process the entire paragraphs.
    # print(tokenizer.model_max_length)

    # model = AutoModelForMaskedLM.from_pretrained("launch/POLITICS")
    model = AutoModelForSequenceClassification.from_pretrained (model_name,num_labels=5) # num_labels = 5 enables hugging face to add a classification head to the model

    train_encodings = tokenizer(x_train_og.to_list(), truncation=True, padding=True, return_tensors="pt")
    train_labels = torch.tensor(y_train, dtype=torch.float32)
    train_dataset = TensorDataset(train_encodings.input_ids, train_encodings.attention_mask, train_labels)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

    # Dataloader for test data
    test_encodings = tokenizer(x_test_og.to_list(), truncation=True, padding=True, return_tensors="pt")
    test_labels = torch.tensor(y_test, dtype=torch.float32)
    test_dataset = TensorDataset(test_encodings.input_ids, test_encodings.attention_mask, test_labels)
    test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)  # No need to shuffle test data

    return train_loader, test_loader, model, tokenizer, mlb.classes_


def evaluate(test_loader, model, tokenizer, classes=None, report=False):
    # Predict on the test data
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for batch in test_loader:
            input_ids, attention_mask, labels = batch
            input_ids, attention_mask, labels = input_ids.to(device), attention_mask.to(device), labels.to(device)

            outputs = model(input_ids, attention_mask=attention_mask)
            logits = outputs.logits

            # Softmax makes more sense for single classifications
            predictions = outputs.logits.softmax(dim=-1).tolist()
            all_preds.extend(predictions)

            # In case you'd want to use Sigmoid
            # predictions = torch.sigmoid(logits)  # Apply sigmoid activation for multilabel classification
            # all_preds.extend(predictions.cpu().detach().numpy())

            all_labels.extend(labels.cpu().detach().numpy())

    # Convert the predictions and labels to binary values based on a threshold (e.g., 0.5)
    threshold = 0.5

    all_preds = (torch.tensor(all_preds) > threshold).int().numpy()
    all_labels = np.array(all_labels)

    # Compute the classification report
    accuracy = accuracy_score(all_labels, all_preds)

    # Reporting Results
    if report:
      #Bigger report summary. Sample avg is the same as Accuracy.
      report = classification_report(all_labels, all_preds, target_names=classes)
      print(report)

    # return more things want more information
    return accuracy


def train(train_loader, test_loader, model, tokenizer, lr, batch_size):
    #Optimizer
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

    # # Fine-tuning loop
    model.to(device)

    num_epochs = 50

    for epoch in range(num_epochs):
        model.train()
        total_loss = 0

        for batch in train_loader:
            input_ids, attention_mask, labels = batch
            input_ids, attention_mask, labels = input_ids.to(device), attention_mask.to(device), labels.to(device)

            optimizer.zero_grad()

            # Forward pass
            outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss

            # # Backward pass and optimization
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        train_acc = evaluate(train_loader, model, tokenizer)
        val_acc = evaluate(test_loader, model, tokenizer)

        print(f"train_acc: {train_acc}")
        print(f"val_acc: {val_acc}")


        wandb.log({
            'loss': total_loss,
            'train_acc': train_acc,
            'val_acc': val_acc,
          })

        print(f"Epoch {epoch + 1}/{num_epochs}, Loss: {total_loss}")
        # test_model() Use if you would want to take a look at how the model is currently doing on the test data. BAD PRACTICE!
        if total_loss < 2:
          break

    return model, tokenizer

# One-off training
This section is for if you just want to train a single model with a given configuration. Record your configuration in the following wandb config, and simply run the training block. The loss will be reported to wandb.

In [5]:
lr = 5e-5
batch_size = 16
test_size = 0.1

wandb.init(
    project='test2',
    config= {
        'learning_rate': lr,
        'batch_size': batch_size,
        'test_size': test_size,
    }
)

wandb: Currently logged in as: ellieyhc (probgram). Use `wandb login --relogin` to force relogin


In [6]:
# Load model directly
train_loader, test_loader, model, tokenizer, classes = init_data_model(batch_size, test_size)

Labels : [['center'], ['right'], ['left'], ['center'], ['center'], ['center'], ['center'], ['liberal'], ['left'], ['center'], ['center'], ['right'], ['center'], ['center'], ['conservative'], ['liberal'], ['center'], ['center'], ['center'], ['liberal'], ['right'], ['center'], ['center'], ['center'], ['center'], ['liberal'], ['conservative'], ['right'], ['liberal'], ['left'], ['right'], ['liberal'], ['center'], ['liberal'], ['center'], ['liberal'], ['conservative'], ['center'], ['center'], ['right'], ['center'], ['right'], ['center'], ['center'], ['center'], ['center'], ['center'], ['liberal'], ['center'], ['center'], ['liberal'], ['center'], ['liberal'], ['conservative'], ['center'], ['liberal'], ['left'], ['center'], ['center'], ['center'], ['conservative'], ['conservative'], ['right'], ['center'], ['left'], ['right'], ['center'], ['liberal'], ['right'], ['right'], ['liberal'], ['center'], ['right'], ['left'], ['right'], ['liberal'], ['conservative'], ['liberal'], ['center'], ['center'

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at launch/POLITICS and are newly initialized: ['classifier.out_proj.bias', 'classifier.out_proj.weight', 'classifier.dense.weight', 'classifier.dense.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [7]:
model, tokenizer = train(train_loader, test_loader, model, tokenizer, lr, batch_size)

Model Accuracy : 0.36666666666666664
Model Accuracy : 0.37037037037037035
Epoch 1/50, Loss: 8.39603441953659
Model Accuracy : 0.43333333333333335
Model Accuracy : 0.48518518518518516
Epoch 2/50, Loss: 7.525094002485275
Model Accuracy : 0.43333333333333335
Model Accuracy : 0.48148148148148145
Epoch 3/50, Loss: 7.341296523809433
Model Accuracy : 0.43333333333333335
Model Accuracy : 0.4740740740740741
Epoch 4/50, Loss: 7.118106931447983
Model Accuracy : 0.43333333333333335
Model Accuracy : 0.5370370370370371
Epoch 5/50, Loss: 6.191679865121841
Model Accuracy : 0.5
Model Accuracy : 0.7296296296296296
Epoch 6/50, Loss: 5.013858452439308
Model Accuracy : 0.5333333333333333
Model Accuracy : 0.8407407407407408
Epoch 7/50, Loss: 4.052236467599869
Model Accuracy : 0.5
Model Accuracy : 0.8518518518518519
Epoch 8/50, Loss: 2.862770065665245
Model Accuracy : 0.5
Model Accuracy : 0.9777777777777777
Epoch 9/50, Loss: 2.0871654972434044
Model Accuracy : 0.5666666666666667
Model Accuracy : 0.9888888888

In [9]:
# final evaluation
acc = evaluate(test_loader, model, tokenizer, classes, report=True)

Model Accuracy : 0.5666666666666667
              precision    recall  f1-score   support

      center       0.52      1.00      0.68        13
conservative       1.00      0.50      0.67         2
        left       0.00      0.00      0.00         3
     liberal       1.00      0.14      0.25         7
       right       0.67      0.40      0.50         5

   micro avg       0.57      0.57      0.57        30
   macro avg       0.64      0.41      0.42        30
weighted avg       0.64      0.57      0.48        30
 samples avg       0.57      0.57      0.57        30



/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


In [10]:
# save the model
save_name = 'test_best'

model.save_pretrained(save_name)

In [11]:
wandb.finish()

loss,█▇▇▇▆▅▄▃▂▁
train_acc,▁▂▂▂▃▅▆▆██
val_acc,▁▃▃▃▃▆▇▆▆█
loss,1.31669
train_acc,0.98889
val_acc,0.56667


# Hyperparameter Fine-tuning
This section is for doing sweeps over different hyperparameters to fine-tune for the best accuracy.

In [9]:
sweep_config = {
    'method': 'random',
    'name': 'sweep',
    'metric': {'goal': 'maximize', 'name': 'val_acc'},
    'parameters': {
        'batch_size': {'values': [8, 16]},
        'lr': {'max': 0.001, 'min': 1e-5},
        'test_size': {'values': [0.1]}
    }
}

sweep_id = wandb.sweep(sweep=sweep_config, project='politics-sweep')

Create sweep with ID: 1vilg9re
Sweep URL: https://wandb.ai/probgram/politics-sweep/sweeps/1vilg9re


In [23]:
def main():
  run = wandb.init()

  lr = wandb.config.lr
  batch_size = wandb.config.batch_size
  test_size = wandb.config.test_size

  train_loader, test_loader, model, tokenizer, classes = init_data_model(batch_size, test_size)
  model, tokenizer = train(train_loader, test_loader, model, tokenizer, lr, batch_size)

  # del test_labels
  del model
  del tokenizer
  torch.cuda.empty_cache()


In [ ]:
wandb.agent(sweep_id, function=main, count=4)

wandb: Agent Starting Run: k2c4fpdc with config:
wandb: 	batch_size: 16
wandb: 	lr: 0.0003547868742193005
wandb: 	test_size: 0.1


Labels : [['center'], ['right'], ['left'], ['center'], ['center'], ['center'], ['center'], ['liberal'], ['left'], ['center'], ['center'], ['right'], ['center'], ['center'], ['conservative'], ['liberal'], ['center'], ['center'], ['center'], ['liberal'], ['right'], ['center'], ['center'], ['center'], ['center'], ['liberal'], ['conservative'], ['right'], ['liberal'], ['left'], ['right'], ['liberal'], ['center'], ['liberal'], ['center'], ['liberal'], ['conservative'], ['center'], ['center'], ['right'], ['center'], ['right'], ['center'], ['center'], ['center'], ['center'], ['center'], ['liberal'], ['center'], ['center'], ['liberal'], ['center'], ['liberal'], ['conservative'], ['center'], ['liberal'], ['left'], ['center'], ['center'], ['center'], ['conservative'], ['conservative'], ['right'], ['center'], ['left'], ['right'], ['center'], ['liberal'], ['right'], ['right'], ['liberal'], ['center'], ['right'], ['left'], ['right'], ['liberal'], ['conservative'], ['liberal'], ['center'], ['center'

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at launch/POLITICS and are newly initialized: ['classifier.out_proj.weight', 'classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


train_acc: 0.48518518518518516
val_acc: 0.43333333333333335
Epoch 1/50, Loss: 7.972112536430359
train_acc: 0.43703703703703706
val_acc: 0.4
Epoch 2/50, Loss: 7.726129591464996
train_acc: 0.48148148148148145
val_acc: 0.43333333333333335
Epoch 3/50, Loss: 7.686459630727768


In [ ]:
wandb.finish()

# Free up memory

In [ ]:
#Memory Management
del df
# del test_labels
del model
del tokenizer
torch.cuda.empty_cache()